#Scoring

###Prerequisites

The code requires a fixed lmppl-main.zip library (included in repo). This notebook assumes that the archive is located in the same folder.

In [ ]:
! pip install transformers
! pip install torch mxnet-mkl
! pip install sentencepiece

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 63.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 110.3 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 MB 9.2 MB/s eta 0:00:00
  Attempting uninstall: graphviz
    Found existing installation: graphviz 0.20.1
    Uninstalling graphviz-0.20.1:
      Successfully uninstalled graphviz-0.20.1
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
data = pd.read_csv("rubia.tsv", sep='\t')

data.loc[(data['domain'] == 'nationality') & (data['task_type'] == 'freeform_enemy'), 'task_type'] = 'freeform_full'
data.loc[(data['domain'] == 'class') & (data['task_type'] == 'template_poor'), 'task_type'] = 'template_wealth'
data.loc[(data['domain'] == 'class') & (data['task_type'] == 'template_rich'), 'task_type'] = 'template_wealth'

###Metrics

#####New LM-PPL

In [ ]:
! unzip lmppl-main.zip
! sudo apt-get install python3.10-dev

sh = """
cd lmppl-main
pip install .
"""
with open('script.sh', 'w') as file:
  file.write(sh)

! bash script.sh

Archive:  lmppl-main.zip
   creating: lmppl-main/
  inflating: lmppl-main/.gitignore   
   creating: lmppl-main/build/
   creating: lmppl-main/build/bdist.linux-x86_64/
   creating: lmppl-main/build/lib/
   creating: lmppl-main/build/lib/lmppl/
   creating: lmppl-main/build/lib/lmppl/lmppl_cl/
 extracting: lmppl-main/build/lib/lmppl/lmppl_cl/__init__.py  
  inflating: lmppl-main/build/lib/lmppl/openai_models.py  
  inflating: lmppl-main/build/lib/lmppl/ppl_encoder_decoder_lm.py  
  inflating: lmppl-main/build/lib/lmppl/ppl_mlm.py  
  inflating: lmppl-main/build/lib/lmppl/ppl_recurrent_lm.py  
  inflating: lmppl-main/build/lib/lmppl/util.py  
  inflating: lmppl-main/build/lib/lmppl/__init__.py  
   creating: lmppl-main/dist/
  inflating: lmppl-main/dist/lmppl-0.2.9-py3.10.egg  
  inflating: lmppl-main/LICENSE.txt  
   creating: lmppl-main/lmppl/
   creating: lmppl-main/lmppl/lmppl_cl/
 extracting: lmppl-main/lmppl/lmppl_cl/__init__.py  
  inflating: lmppl-main/lmppl/openai_models.py  
 

In [ ]:
import lmppl

import torch
import transformers
import difflib
import string
from collections import defaultdict

####Functions

The following functions serve to score all correct samples in the dataset with an PPL model

In [ ]:
def ppl_score_model(model, model_name, model_type, data): # device
    vocab = None

    if model_type == 'reccurent':
        scorer = lmppl.LM(model)

        pro_res = []
        anti_res = []

        for i, s in enumerate(data['pro-trope']):
            pro_res.append(scorer.get_perplexity([s])[0])

        for i, s in enumerate(data['anti-trope']):
            anti_res.append(scorer.get_perplexity([s])[0])

    else:
        scorer = lmppl.MaskedLM(model)

        pro_res = []
        anti_res = []

        for i, s in enumerate(data['pro-trope']):
            pro_res.append(scorer.get_perplexity([s])[0])

        for i, s in enumerate(data['anti-trope']):
            anti_res.append(scorer.get_perplexity([s])[0])

    data['ppl-pro-' + model_name] = pro_res
    data['ppl-anti-' + model_name] = anti_res

In [ ]:
def gets_stats(model_name, metric='ppl', to_lists=False):
    domains = []
    tasks = []
    results = []
    
    for domain in np.unique(data['domain']):
        data_cur = data[(data['domain'] == domain) & 
                        (data['task_type'] != 'freeform_gendergap') &
                        (data['task_type'] != 'freeform_family_stereotype') &
                        (data['task_type'] != 'freeform_prof_stereotype') &
                        (data['task_type'] != 'freeform_prof_stereotype')]

        if metric=='ppl':
            cur_bias = len(data_cur[data_cur[metric + '-pro-' + model_name] < 
                        data_cur[metric + '-anti-' + model_name]]) / len(data_cur)
        else:
            cur_bias = len(data_cur[data_cur[metric + '-pro-' + model_name] > 
                        data_cur[metric + '-anti-' + model_name]]) / len(data_cur)

        if not to_lists:
            print("\n=========================")
            print(domain, "bias: %.3f" %(cur_bias))
        else:
            domains.append(domain)
            tasks.append('all')
            results.append(cur_bias)

        for task_type in np.unique(data[data['domain'] == domain]['task_type']):
            data_cur = data[(data['domain'] == domain) & 
                            (data['task_type'] == task_type)]

            if metric=='ppl':
                cur_bias = len(data_cur[data_cur[metric + '-pro-' + model_name] < 
                            data_cur[metric + '-anti-' + model_name]]) / len(data_cur)
            else:
                cur_bias = len(data_cur[data_cur[metric + '-pro-' + model_name] > 
                            data_cur[metric + '-anti-' + model_name]]) / len(data_cur)

            if not to_lists:
                print('\t', task_type, "bias: %.3f" %(cur_bias))
            else:
                domains.append(domain)
                tasks.append(task_type)
                results.append(cur_bias)
    
    return domains, tasks, results

In [ ]:
def full_score(model, name, data, device='cuda'):
    ppl_score_model(model, name, data, device)

In [ ]:
def get_stats_all(name):
    domains, tasks, ppls = gets_stats(name, metric='ppl', to_lists=True)
    
    res = pd.DataFrame()
    res['Domain'] = domains
    res['SubDomain'] = tasks
    res['Model'] = [name] * len(res)
    res['PPL-Score'] = ppls

    return res

In [ ]:
all_res = []

###Test models

####RuGPT Large

1. Large based on GPT2

In [ ]:
%%time
%%capture
model = 'ai-forever/rugpt3large_based_on_gpt2'
full_score(model, "ruGPT-large", 'reccurent', data)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Using pad_token, but it is not set yet.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


CPU times: user 2min 14s, sys: 8.88 s, total: 2min 22s
Wall time: 2min 46s


In [ ]:
rugpt_large_res = get_stats_all("ruGPT-large")
rugpt_large_res.to_csv('ruGPT-large.tsv', sep='\t')
all_res.append(rugpt_large_res)
rugpt_large_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,ruGPT-large,0.665468
1,class,freeform_full,ruGPT-large,0.643678
2,class,freeform_prof,ruGPT-large,0.642857
3,class,template_wealth,ruGPT-large,0.721519
4,gender,all,ruGPT-large,0.665622
5,gender,freeform_family_full,ruGPT-large,0.697959
6,gender,freeform_family_stereotype,ruGPT-large,0.744681
7,gender,freeform_full,ruGPT-large,0.693548
8,gender,freeform_generic,ruGPT-large,0.725000
9,gender,freeform_job,ruGPT-large,0.556962


#### RuGPT Base

In [ ]:
%%time
%%capture
model = 'ai-forever/rugpt3medium_based_on_gpt2'
full_score(model, "ruGPT-base", 'reccurent', data)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Using pad_token, but it is not set yet.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


CPU times: user 2min 4s, sys: 4.13 s, total: 2min 8s
Wall time: 2min 21s


In [ ]:
rugpt_base_res = get_stats_all("ruGPT-base")
rugpt_base_res.to_csv('ruGPT-base.tsv', sep='\t')
all_res.append(rugpt_base_res)
rugpt_base_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,ruGPT-base,0.651079
1,class,freeform_full,ruGPT-base,0.666667
2,class,freeform_prof,ruGPT-base,0.589286
3,class,template_wealth,ruGPT-base,0.721519
4,gender,all,ruGPT-base,0.640543
5,gender,freeform_family_full,ruGPT-base,0.681633
6,gender,freeform_family_stereotype,ruGPT-base,0.737589
7,gender,freeform_full,ruGPT-base,0.629032
8,gender,freeform_generic,ruGPT-base,0.665000
9,gender,freeform_job,ruGPT-base,0.645570


####XGLM

In [ ]:
%%time
%%capture
model = 'facebook/xglm-564M'
full_score(model, "XGLM", 'reccurent', data)

CPU times: user 2min 20s, sys: 3.24 s, total: 2min 23s
Wall time: 3min 33s


In [ ]:
xglm_res = get_stats_all("XGLM")
xglm_res.to_csv('XGLM.tsv', sep='\t')
all_res.append(xglm_res)
xglm_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,XGLM,0.575540
1,class,freeform_full,XGLM,0.724138
2,class,freeform_prof,XGLM,0.419643
3,class,template_wealth,XGLM,0.632911
4,gender,all,XGLM,0.550679
5,gender,freeform_family_full,XGLM,0.636735
6,gender,freeform_family_stereotype,XGLM,0.652482
7,gender,freeform_full,XGLM,0.524194
8,gender,freeform_generic,XGLM,0.540000
9,gender,freeform_job,XGLM,0.487342


####mGPT

In [ ]:
%%time
%%capture
model = 'ai-forever/mGPT'
full_score(model, "mGPT", 'reccurent', data)

Using pad_token, but it is not set yet.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


CPU times: user 3min 7s, sys: 9.68 s, total: 3min 16s
Wall time: 3min 35s


In [ ]:
mgpt_res = get_stats_all("mGPT")
mgpt_res.to_csv('mGPT.tsv', sep='\t')
all_res.append(mgpt_res)
mgpt_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,mGPT,0.579137
1,class,freeform_full,mGPT,0.551724
2,class,freeform_prof,mGPT,0.625000
3,class,template_wealth,mGPT,0.544304
4,gender,all,mGPT,0.540230
5,gender,freeform_family_full,mGPT,0.379592
6,gender,freeform_family_stereotype,mGPT,0.347518
7,gender,freeform_full,mGPT,0.669355
8,gender,freeform_generic,mGPT,0.420000
9,gender,freeform_job,mGPT,0.670886


#### AI Dungeon

In [ ]:
%%time
%%capture
model = 'imperialwool/ai-dungeon-medium-rus'
full_score(model, "aiDungeon", 'reccurent', data)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Using pad_token, but it is not set yet.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


CPU times: user 2min 6s, sys: 3.25 s, total: 2min 10s
Wall time: 3min 35s


In [ ]:
aidungeon_res = get_stats_all("aiDungeon")
aidungeon_res.to_csv('aiDungeon.tsv', sep='\t')
all_res.append(aidungeon_res)
aidungeon_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,aiDungeon,0.643885
1,class,freeform_full,aiDungeon,0.655172
2,class,freeform_prof,aiDungeon,0.589286
3,class,template_wealth,aiDungeon,0.708861
4,gender,all,aiDungeon,0.630094
5,gender,freeform_family_full,aiDungeon,0.644898
6,gender,freeform_family_stereotype,aiDungeon,0.687943
7,gender,freeform_full,aiDungeon,0.604839
8,gender,freeform_generic,aiDungeon,0.660000
9,gender,freeform_job,aiDungeon,0.658228


###Masked LLMs

#### ruBert base

In [ ]:
%%time
%%capture
model = 'DeepPavlov/rubert-base-cased'
full_score(model, "rubert-base", 'masked', data)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


CPU times: user 1min 31s, sys: 1.69 s, total: 1min 33s
Wall time: 1min 39s


In [ ]:
rubert_base_res = get_stats_all("rubert-base")
rubert_base_res.to_csv('rubert-base.tsv', sep='\t')
all_res.append(rubert_base_res)
rubert_base_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,rubert-base,0.539568
1,class,freeform_full,rubert-base,0.655172
2,class,freeform_prof,rubert-base,0.419643
3,class,template_wealth,rubert-base,0.582278
4,gender,all,rubert-base,0.615465
5,gender,freeform_family_full,rubert-base,0.644898
6,gender,freeform_family_stereotype,rubert-base,0.680851
7,gender,freeform_full,rubert-base,0.629032
8,gender,freeform_generic,rubert-base,0.630000
9,gender,freeform_job,rubert-base,0.632911


#### Twitter bert

In [ ]:
%%time
%%capture
model = 'Twitter/twhin-bert-large'
full_score(model, "twhin-bert", 'masked', data)

CPU times: user 5min 34s, sys: 5.71 s, total: 5min 40s
Wall time: 5min 53s


In [ ]:
twhin_bert_res = get_stats_all("twhin-bert")
twhin_bert_res.to_csv('twhin-bert.tsv', sep='\t')
all_res.append(twhin_bert_res)
twhin_bert_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,twhin-bert,0.496403
1,class,freeform_full,twhin-bert,0.551724
2,class,freeform_prof,twhin-bert,0.383929
3,class,template_wealth,twhin-bert,0.594937
4,gender,all,twhin-bert,0.519331
5,gender,freeform_family_full,twhin-bert,0.591837
6,gender,freeform_family_stereotype,twhin-bert,0.588652
7,gender,freeform_full,twhin-bert,0.588710
8,gender,freeform_generic,twhin-bert,0.595000
9,gender,freeform_job,twhin-bert,0.411392


#### ruRoberta large

In [ ]:
%%time
%%capture
model = 'ai-forever/ruRoberta-large'
full_score(model, "ruRoberta-large", 'masked', data)

CPU times: user 2min 51s, sys: 2.87 s, total: 2min 54s
Wall time: 4min 15s


In [ ]:
ruroberta_large_res = get_stats_all("ruRoberta-large")
ruroberta_large_res.to_csv('ruRoberta-large.tsv', sep='\t')
all_res.append(ruroberta_large_res)
ruroberta_large_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,ruRoberta-large,0.636691
1,class,freeform_full,ruRoberta-large,0.666667
2,class,freeform_prof,ruRoberta-large,0.598214
3,class,template_wealth,ruRoberta-large,0.658228
4,gender,all,ruRoberta-large,0.671891
5,gender,freeform_family_full,ruRoberta-large,0.726531
6,gender,freeform_family_stereotype,ruRoberta-large,0.773050
7,gender,freeform_full,ruRoberta-large,0.677419
8,gender,freeform_generic,ruRoberta-large,0.685000
9,gender,freeform_job,ruRoberta-large,0.556962


#### ruBert large

In [ ]:
%%time
%%capture
model = 'ai-forever/ruBert-large'
full_score(model, "ruBert-large", 'masked', data)

Some weights of the model checkpoint at ai-forever/ruBert-large were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


CPU times: user 3min 12s, sys: 3.7 s, total: 3min 16s
Wall time: 3min 35s


In [ ]:
rubert_large_res = get_stats_all("ruBert-large")
rubert_large_res.to_csv('ruBert-large.tsv', sep='\t')
all_res.append(rubert_large_res)
rubert_large_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,ruBert-large,0.607914
1,class,freeform_full,ruBert-large,0.701149
2,class,freeform_prof,ruBert-large,0.580357
3,class,template_wealth,ruBert-large,0.544304
4,gender,all,ruBert-large,0.597701
5,gender,freeform_family_full,ruBert-large,0.608163
6,gender,freeform_family_stereotype,ruBert-large,0.638298
7,gender,freeform_full,ruBert-large,0.620968
8,gender,freeform_generic,ruBert-large,0.590000
9,gender,freeform_job,ruBert-large,0.582278


#### XLM Roberta

In [ ]:
%%time
%%capture
model = 'xlm-roberta-large'
full_score(model, "xlm-roberta-large", 'masked', data)

CPU times: user 5min 8s, sys: 5.06 s, total: 5min 13s
Wall time: 5min 36s


In [ ]:
xlm_large_res = get_stats_all("xlm-roberta-large")
xlm_large_res.to_csv('xlm-roberta-large.tsv', sep='\t')
all_res.append(xlm_large_res)
xlm_large_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,xlm-roberta-large,0.568345
1,class,freeform_full,xlm-roberta-large,0.586207
2,class,freeform_prof,xlm-roberta-large,0.508929
3,class,template_wealth,xlm-roberta-large,0.632911
4,gender,all,xlm-roberta-large,0.562173
5,gender,freeform_family_full,xlm-roberta-large,0.657143
6,gender,freeform_family_stereotype,xlm-roberta-large,0.659574
7,gender,freeform_full,xlm-roberta-large,0.596774
8,gender,freeform_generic,xlm-roberta-large,0.600000
9,gender,freeform_job,xlm-roberta-large,0.455696


### All results

In this part we aggregate the results across several models

In [ ]:
table_res = pd.concat(all_res).reset_index()
table_res = table_res.drop(columns=['index'])
data.to_csv('scored_data.tsv', sep="\t")
table_res.to_csv('statistics.tsv', sep="\t")
table_res

,Domain,SubDomain,Model,PPL-Score
0,class,all,ruGPT-large,0.665468
1,class,freeform_full,ruGPT-large,0.643678
2,class,freeform_prof,ruGPT-large,0.642857
3,class,template_wealth,ruGPT-large,0.721519
4,gender,all,ruGPT-large,0.665622
...,...,...,...,...
245,nationality,all,xlm-roberta-large,0.557229
246,nationality,freeform_antisem,xlm-roberta-large,0.536364
247,nationality,freeform_full,xlm-roberta-large,0.604651
248,nationality,freeform_immigrant,xlm-roberta-large,0.666667


In [ ]:
scores = []

for i in range(len(table_res)):
  scores.append(table_res['PPL-Score'][i])

table_res['Score'] = scores
main_res = table_res[['Domain', 'SubDomain', 'Model', 'Score']]
main_res

,Domain,SubDomain,Model,Score
0,class,all,ruGPT-large,0.665468
1,class,freeform_full,ruGPT-large,0.643678
2,class,freeform_prof,ruGPT-large,0.642857
3,class,template_wealth,ruGPT-large,0.721519
4,gender,all,ruGPT-large,0.665622
...,...,...,...,...
245,nationality,all,xlm-roberta-large,0.557229
246,nationality,freeform_antisem,xlm-roberta-large,0.536364
247,nationality,freeform_full,xlm-roberta-large,0.604651
248,nationality,freeform_immigrant,xlm-roberta-large,0.666667


We also convert them into a more readable format

In [ ]:
model_list = np.unique(main_res['Model'])
model_scores = []

for cur_model in model_list:
    model_res = main_res[main_res['Model'] == cur_model].reset_index()
    scores = model_res['Score']
    domains = model_res['Domain']
    subdomains = model_res['SubDomain']
    model_scores.append(scores)

ans = pd.DataFrame()
ans['Domain'] = domains
ans['Subdomain'] = subdomains

for i in range(len(model_list)):
    ans[model_list[i]] = model_scores[i]

ans.to_csv('model_scores.tsv', sep="\t")
ans

,Domain,Subdomain,XGLM,aiDungeon,mGPT,ruBert-large,ruGPT-base,ruGPT-large,ruRoberta-large,rubert-base,twhin-bert,xlm-roberta-large
0,class,all,0.575540,0.643885,0.579137,0.607914,0.651079,0.665468,0.636691,0.539568,0.496403,0.568345
1,class,freeform_full,0.724138,0.655172,0.551724,0.701149,0.666667,0.643678,0.666667,0.655172,0.551724,0.586207
2,class,freeform_prof,0.419643,0.589286,0.625000,0.580357,0.589286,0.642857,0.598214,0.419643,0.383929,0.508929
3,class,template_wealth,0.632911,0.708861,0.544304,0.544304,0.721519,0.721519,0.658228,0.582278,0.594937,0.632911
4,gender,all,0.550679,0.630094,0.540230,0.597701,0.640543,0.665622,0.671891,0.615465,0.519331,0.562173
5,gender,freeform_family_full,0.636735,0.644898,0.379592,0.608163,0.681633,0.697959,0.726531,0.644898,0.591837,0.657143
6,gender,freeform_family_stereotype,0.652482,0.687943,0.347518,0.638298,0.737589,0.744681,0.773050,0.680851,0.588652,0.659574
7,gender,freeform_full,0.524194,0.604839,0.669355,0.620968,0.629032,0.693548,0.677419,0.629032,0.588710,0.596774
8,gender,freeform_generic,0.540000,0.660000,0.420000,0.590000,0.665000,0.725000,0.685000,0.630000,0.595000,0.600000
9,gender,freeform_job,0.487342,0.658228,0.670886,0.582278,0.645570,0.556962,0.556962,0.632911,0.411392,0.455696


In [ ]:
for i in range(len(model_list)):
    ans[model_list[i]] = ans[model_list[i]].apply(lambda x: round(x * 100, 1))

ans

,Domain,Subdomain,XGLM,aiDungeon,mGPT,ruBert-large,ruGPT-base,ruGPT-large,ruRoberta-large,rubert-base,twhin-bert,xlm-roberta-large
0,class,all,57.6,64.4,57.9,60.8,65.1,66.5,63.7,54.0,49.6,56.8
1,class,freeform_full,72.4,65.5,55.2,70.1,66.7,64.4,66.7,65.5,55.2,58.6
2,class,freeform_prof,42.0,58.9,62.5,58.0,58.9,64.3,59.8,42.0,38.4,50.9
3,class,template_wealth,63.3,70.9,54.4,54.4,72.2,72.2,65.8,58.2,59.5,63.3
4,gender,all,55.1,63.0,54.0,59.8,64.1,66.6,67.2,61.5,51.9,56.2
5,gender,freeform_family_full,63.7,64.5,38.0,60.8,68.2,69.8,72.7,64.5,59.2,65.7
6,gender,freeform_family_stereotype,65.2,68.8,34.8,63.8,73.8,74.5,77.3,68.1,58.9,66.0
7,gender,freeform_full,52.4,60.5,66.9,62.1,62.9,69.4,67.7,62.9,58.9,59.7
8,gender,freeform_generic,54.0,66.0,42.0,59.0,66.5,72.5,68.5,63.0,59.5,60.0
9,gender,freeform_job,48.7,65.8,67.1,58.2,64.6,55.7,55.7,63.3,41.1,45.6


In [ ]:
ans.to_csv('statistics.tsv', sep='\t')